In [0]:
# Création du catalogue
spark.sql("""
CREATE CATALOG IF NOT EXISTS divvy
""")

# Création du schema Bronze
spark.sql("""
CREATE SCHEMA IF NOT EXISTS divvy.bronze
""")

# Nom de la table Bronze
BRONZE_TABLE = "divvy.bronze.trips_raw"

In [0]:

spark.sql('CREATE CATALOG IF NOT EXISTS divvy_catalog');

spark.sql('CREATE SCHEMA IF NOT EXISTS divvy_catalog.raw');

spark.sql('CREATE SCHEMA IF NOT EXISTS divvy_catalog.bronze');



In [0]:
RAW_PATH = "/Volumes/divvy_catalog/raw/divvy_trips/"

dbutils.fs.ls(RAW_PATH)

[]

In [0]:
test_file = (
    "/Volumes/divvy_catalog/raw/divvy_trips/"
    "test.json"
)

dbutils.fs.put(
    test_file,
    '{"test": "Databricks"}',
    overwrite=True
)

print("Écriture réussie")

Wrote 22 bytes.
Écriture réussie


In [0]:
dbutils.fs.ls(
    "/Volumes/divvy_catalog/raw/divvy_trips/"
)

[FileInfo(path='dbfs:/Volumes/divvy_catalog/raw/divvy_trips/test.json', name='test.json', size=22, modificationTime=1787698299000)]

In [0]:
# ============================================================
# DIVVY API → DATA LAKE RAW → BRONZE
# ============================================================

# ============================================================
# 1. IMPORTATION DES BIBLIOTHÈQUES
# ============================================================

# Permet d'envoyer des requêtes HTTP vers l'API
import requests

# Permet de convertir les données Python en JSON
import json

# Permet de récupérer la date et l'heure de l'extraction
from datetime import datetime

# Fonctions Spark
from pyspark.sql import functions as F


# ============================================================
# 2. CONFIGURATION DE L'API
# ============================================================

# URL de l'API Divvy Trips de Chicago
API_URL = (
    "https://data.cityofchicago.org/"
    "resource/fg6s-gzvg.json"
)


# ============================================================
# 3. PARAMÈTRES DE L'API
# ============================================================

# Nombre de lignes que nous souhaitons récupérer
LIMIT = 1000

# Paramètres envoyés à l'API Socrata
params = {
    "$limit": LIMIT,
    "$order": "trip_id"
}


# ============================================================
# 4. CONFIGURATION DU DATA LAKE
# ============================================================

# Emplacement dans lequel nous allons sauvegarder
# les données brutes de l'API
RAW_PATH = (
    "/Volumes/divvy_catalog/raw/divvy_trips/"
)


# ============================================================
# 5. CONFIGURATION DE LA TABLE BRONZE
# ============================================================

# Nom complet de la table Bronze
#
# catalog.schema.table
#
# divvy_catalog = catalogue
# bronze        = schema
# divvy_trips_raw = table
BRONZE_TABLE = (
    "divvy_catalog.bronze.divvy_trips_raw"
)


# ============================================================
# 6. APPEL DE L'API
# ============================================================

print("Début de l'extraction API...")

response = requests.get(
    API_URL,

    # IMPORTANT :
    # On transmet réellement les paramètres
    # $limit et $order à l'API.
    params=params,

    # Temps maximum d'attente de la réponse.
    timeout=120
)


# Affichage du statut HTTP
print(
    f"Status HTTP : {response.status_code}"
)


# Affichage de l'URL réellement appelée
print(
    f"URL appelée : {response.url}"
)


# ============================================================
# 7. VALIDATION DE LA RÉPONSE
# ============================================================

# HTTP 200 signifie que la requête a réussi.
if response.status_code != 200:

    raise Exception(
        f"Erreur API : HTTP "
        f"{response.status_code}"
    )


# ============================================================
# 8. RÉCUPÉRATION DU JSON
# ============================================================

# Conversion de la réponse HTTP en objet Python.
#
# IMPORTANT :
# L'API Divvy renvoie directement une LISTE.
#
# Exemple :
#
# [
#     {
#         "trip_id": "25962904",
#         "start_time": "...",
#         ...
#     },
#     {
#         "trip_id": "25962903",
#         ...
#     }
# ]
#
# Elle ne renvoie PAS :
#
# {
#     "data": [...]
# }

payload = response.json()


# ============================================================
# 9. VÉRIFICATION DU FORMAT
# ============================================================

# Nous vérifions que la réponse est bien une liste.
if not isinstance(payload, list):

    raise Exception(
        "La réponse API n'est pas une "
        "liste de données."
    )


# Vérification qu'il y a effectivement des données.
if len(payload) == 0:

    raise Exception(
        "L'API n'a retourné aucune donnée."
    )


# Affichage du nombre de lignes récupérées
print(
    f"Nombre de lignes récupérées : "
    f"{len(payload)}"
)


# ============================================================
# 10. IDENTIFICATION DE L'EXTRACTION
# ============================================================

# Récupération de la date et de l'heure actuelles
extraction_timestamp = datetime.utcnow()


# Format de la date
# Exemple : 2026-08-25
extraction_date = (
    extraction_timestamp.strftime(
        "%Y-%m-%d"
    )
)


# Format de l'heure
# Exemple : 213045
extraction_time = (
    extraction_timestamp.strftime(
        "%H%M%S"
    )
)


# ============================================================
# 11. CRÉATION DU NOM DU FICHIER RAW
# ============================================================

raw_file = (
    f"{RAW_PATH}"
    f"extraction_{extraction_date}_"
    f"{extraction_time}.json"
)


# Affichage du chemin du fichier
print(
    f"Fichier RAW : {raw_file}"
)


# ============================================================
# 12. SAUVEGARDE DU JSON ORIGINAL
#     DANS LE DATA LAKE
# ============================================================

# On sauvegarde exactement la réponse reçue
# depuis l'API.
#
# json.dumps() transforme l'objet Python
# en texte JSON.

dbutils.fs.put(
    raw_file,

    json.dumps(
        payload
    ),

    # False = ne pas écraser un fichier existant
    overwrite=False
)


print(
    "Données RAW sauvegardées avec succès."
)


# ============================================================
# 13. LECTURE DU FICHIER RAW AVEC SPARK
# ============================================================

# Maintenant que les données sont dans le Data Lake,
# Spark lit le fichier JSON.

df = spark.read.json(
    raw_file
)


# ============================================================
# 14. AFFICHAGE DU SCHÉMA
# ============================================================

print(
    "Schéma détecté par Spark :"
)

df.printSchema()


# ============================================================
# 15. AFFICHAGE DES DONNÉES
# ============================================================

display(df)


# ============================================================
# 16. AJOUT DES MÉTADONNÉES TECHNIQUES
# ============================================================

# Création duDataFrame destiné à Bronze.

df_bronze = (
    df

    # Date et heure de l'extraction
    .withColumn(
        "_extracted_at",
        F.current_timestamp()
    )

    # Source des données
    .withColumn(
        "_source",
        F.lit(API_URL)
    )
)


# ============================================================
# 17. AFFICHAGE DU DATAFRAME BRONZE
# ============================================================

display(
    df_bronze
)


# ============================================================
# 18. ÉCRITURE DANS BRONZE
# ============================================================

print(
    "Écriture dans la table Bronze..."
)


(
    df_bronze

    # Utilisation du format Delta Lake
    .write
    .format("delta")

    # Ajout des nouvelles données
    .mode("append")

    # Création/alimentation de la table Unity Catalog
    .saveAsTable(
        BRONZE_TABLE
    )
)


# ============================================================
# 19. CONTRÔLE DE LA TABLE BRONZE
# ============================================================

# Lecture de la table Bronze
bronze_df = spark.table(
    BRONZE_TABLE
)


# Comptage des lignes
count = bronze_df.count()


# ============================================================
# 20. AFFICHAGE DU RÉSULTAT
# ============================================================

print(
    "============================================"
)

print(
    "Pipeline terminé avec succès."
)

print(
    f"Nombre total de lignes Bronze : {count}"
)

print(
    f"Table Bronze : {BRONZE_TABLE}"
)

print(
    "============================================"
)


# ============================================================
# FIN DU PIPELINE
# ============================================================
#
# Le pipeline s'arrête volontairement ici.
#
# API
#  ↓
# Data Lake RAW
#  ↓
# Spark
#  ↓
# Bronze Delta
#  ↓
# STOP
#


Début de l'extraction API...
Status HTTP : 200
URL appelée : https://data.cityofchicago.org/resource/fg6s-gzvg.json?%24limit=1000&%24order=trip_id
Nombre de lignes récupérées : 1000
Fichier RAW : /Volumes/divvy_catalog/raw/divvy_trips/extraction_2026-09-09_083432.json


/home/spark-da8fd47e-6f83-419e-a5e3-d8/.ipykernel/87/command-5055872498979382-950656346:183: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  extraction_timestamp = datetime.utcnow()


Wrote 602390 bytes.
Données RAW sauvegardées avec succès.
Schéma détecté par Spark :
root
 |-- bike_id: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- from_latitude: string (nullable = true)
 |-- from_location: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |-- from_longitude: string (nullable = true)
 |-- from_station_id: string (nullable = true)
 |-- from_station_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- start_time: string (nullable = true)
 |-- stop_time: string (nullable = true)
 |-- to_latitude: string (nullable = true)
 |-- to_location: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |-- to_longitude: string (nullable = true)
 |-- to_station_id: string (nullable = true)
 |-- to_station_nam

bike_id,birth_year,from_latitude,from_location,from_longitude,from_station_id,from_station_name,gender,start_time,stop_time,to_latitude,to_location,to_longitude,to_station_id,to_station_name,trip_duration,trip_id,user_type
914,1982,41.88338,"List(List(-87.64117, 41.88338), Point)",-87.64117,91,Clinton St & Washington Blvd,Male,2013-06-27T01:06:00.000,2013-06-27T09:46:00.000,41.897764,"List(List(-87.642884, 41.897764), Point)",-87.642884,48,Larrabee St & Kingsbury St,31177,3940,Subscriber
480,1982,41.90096039,"List(List(-87.623777, 41.90096), Point)",-87.62377664,85,Michigan Ave & Oak St,Male,2013-06-27T12:06:00.000,2013-06-27T12:11:00.000,41.90096039,"List(List(-87.623777, 41.90096), Point)",-87.62377664,85,Michigan Ave & Oak St,301,4095,Subscriber
711,1982,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,Male,2013-06-27T11:09:00.000,2013-06-27T11:11:00.000,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,140,4113,Subscriber
480,null,41.90096039,"List(List(-87.623777, 41.90096), Point)",-87.62377664,85,Michigan Ave & Oak St,null,2013-06-27T12:11:00.000,2013-06-27T12:16:00.000,41.91468,"List(List(-87.64332, 41.91468), Point)",-87.64332,28,Larrabee St & Menomonee St,316,4118,Customer
711,1982,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,Male,2013-06-27T11:12:00.000,2013-06-27T11:13:00.000,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,87,4119,Subscriber
145,1978,41.90332,"List(List(-87.67273, 41.90332), Point)",-87.67273,17,Wood St & Division St,Male,2013-06-27T11:24:00.000,2013-06-27T14:38:00.000,41.907655,"List(List(-87.672552, 41.907655), Point)",-87.672552,61,Wood St & Milwaukee Ave,11674,4134,Subscriber
711,1982,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,Male,2013-06-27T11:39:00.000,2013-06-27T16:01:00.000,41.926755988,"List(List(-87.634429, 41.926756), Point)",-87.634428785,34,Cannon Dr & Fullerton Ave,15758,4162,Subscriber
303,1982,41.91468,"List(List(-87.64332, 41.91468), Point)",-87.64332,28,Larrabee St & Menomonee St,Male,2013-06-27T12:15:00.000,2013-06-27T12:16:00.000,41.91468,"List(List(-87.64332, 41.91468), Point)",-87.64332,28,Larrabee St & Menomonee St,60,4192,Subscriber
907,1982,41.876243,"List(List(-87.624426, 41.876243), Point)",-87.624426,45,Michigan Ave & Congress Pkwy,Male,2013-06-27T13:00:00.000,2013-06-27T13:03:00.000,41.8810317,"List(List(-87.624084, 41.881032), Point)",-87.62408432,90,Millennium Park,171,4216,Subscriber
907,1982,41.876243,"List(List(-87.624426, 41.876243), Point)",-87.624426,45,Michigan Ave & Congress Pkwy,Male,2013-06-27T13:18:00.000,2013-06-27T19:34:00.000,41.896362458,"List(List(-87.654061, 41.896362), Point)",-87.654061273,54,Ogden Ave & Chicago Ave,22549,4255,Subscriber


bike_id,birth_year,from_latitude,from_location,from_longitude,from_station_id,from_station_name,gender,start_time,stop_time,to_latitude,to_location,to_longitude,to_station_id,to_station_name,trip_duration,trip_id,user_type,_extracted_at,_source
914,1982,41.88338,"List(List(-87.64117, 41.88338), Point)",-87.64117,91,Clinton St & Washington Blvd,Male,2013-06-27T01:06:00.000,2013-06-27T09:46:00.000,41.897764,"List(List(-87.642884, 41.897764), Point)",-87.642884,48,Larrabee St & Kingsbury St,31177,3940,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
480,1982,41.90096039,"List(List(-87.623777, 41.90096), Point)",-87.62377664,85,Michigan Ave & Oak St,Male,2013-06-27T12:06:00.000,2013-06-27T12:11:00.000,41.90096039,"List(List(-87.623777, 41.90096), Point)",-87.62377664,85,Michigan Ave & Oak St,301,4095,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
711,1982,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,Male,2013-06-27T11:09:00.000,2013-06-27T11:11:00.000,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,140,4113,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
480,null,41.90096039,"List(List(-87.623777, 41.90096), Point)",-87.62377664,85,Michigan Ave & Oak St,null,2013-06-27T12:11:00.000,2013-06-27T12:16:00.000,41.91468,"List(List(-87.64332, 41.91468), Point)",-87.64332,28,Larrabee St & Menomonee St,316,4118,Customer,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
711,1982,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,Male,2013-06-27T11:12:00.000,2013-06-27T11:13:00.000,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,87,4119,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
145,1978,41.90332,"List(List(-87.67273, 41.90332), Point)",-87.67273,17,Wood St & Division St,Male,2013-06-27T11:24:00.000,2013-06-27T14:38:00.000,41.907655,"List(List(-87.672552, 41.907655), Point)",-87.672552,61,Wood St & Milwaukee Ave,11674,4134,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
711,1982,41.88397,"List(List(-87.655688, 41.88397), Point)",-87.655688,88,May St & Randolph St,Male,2013-06-27T11:39:00.000,2013-06-27T16:01:00.000,41.926755988,"List(List(-87.634429, 41.926756), Point)",-87.634428785,34,Cannon Dr & Fullerton Ave,15758,4162,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
303,1982,41.91468,"List(List(-87.64332, 41.91468), Point)",-87.64332,28,Larrabee St & Menomonee St,Male,2013-06-27T12:15:00.000,2013-06-27T12:16:00.000,41.91468,"List(List(-87.64332, 41.91468), Point)",-87.64332,28,Larrabee St & Menomonee St,60,4192,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
907,1982,41.876243,"List(List(-87.624426, 41.876243), Point)",-87.624426,45,Michigan Ave & Congress Pkwy,Male,2013-06-27T13:00:00.000,2013-06-27T13:03:00.000,41.8810317,"List(List(-87.624084, 41.881032), Point)",-87.62408432,90,Millennium Park,171,4216,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json
907,1982,41.876243,"List(List(-87.624426, 41.876243), Point)",-87.624426,45,Michigan Ave & Congress Pkwy,Male,2013-06-27T13:18:00.000,2013-06-27T19:34:00.000,41.896362458,"List(List(-87.654061, 41.896362), Point)",-87.654061273,54,Ogden Ave & Chicago Ave,22549,4255,Subscriber,2026-09-09T08:35:12.256Z,https://data.cityofchicago.org/resource/fg6s-gzvg.json


Écriture dans la table Bronze...
Pipeline terminé avec succès.
Nombre total de lignes Bronze : 2000
Table Bronze : divvy_catalog.bronze.divvy_trips_raw
